In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Fault Labels

In [22]:
canopy_2015 = pd.read_csv("/content/drive/MyDrive/NIST_Preprocessed/canopy_2015_Labeled.csv")
canopy_2016 = pd.read_csv("/content/drive/MyDrive/NIST_Preprocessed/canopy_2016_Labeled.csv")

In [24]:
# Fault Indicator Columns
# Rather than merging label_string/label_inverter into a single priority-ordered
# column (which would hide genuine co-occurrence between fault types), we keep
# both label sets fully separate and add lightweight boolean indicators on top.
# This supports both a coarse "fault vs no-fault" detection task and full
# subsystem-level classification, without discarding information.

NON_FAULT_VALUES = ['NORMAL', 'NOT_EVALUABLE', 'MISSING_DATA', 'INSUFFICIENT_DATA']

canopy_2015['has_string_fault']   = ~canopy_2015['label_string'].isin(NON_FAULT_VALUES)
canopy_2015['has_inverter_fault'] = ~canopy_2015['label_inverter'].isin(NON_FAULT_VALUES)
canopy_2015['has_any_fault']      = canopy_2015['has_string_fault'] | canopy_2015['has_inverter_fault']
canopy_2015['fault_cooccurrence'] = canopy_2015['has_string_fault'] & canopy_2015['has_inverter_fault']

# Summary
print("=== Fault Indicator Summary (2015) ===\n")
print(f"has_string_fault   : {canopy_2015['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2015['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2015['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2015['fault_cooccurrence'].sum()}")

print(f"\nAs % of evaluable rows (Op_Mask=1): "
      f"{canopy_2015.loc[canopy_2015['Op_Mask']==1, 'has_any_fault'].mean() * 100:.2f}%")

=== Fault Indicator Summary (2015) ===

has_string_fault   : 11146
has_inverter_fault : 10380
has_any_fault      : 20801
fault_cooccurrence : 725

As % of evaluable rows (Op_Mask=1): 10.92%


In [25]:
NON_FAULT_VALUES = ['NORMAL', 'NOT_EVALUABLE', 'MISSING_DATA', 'INSUFFICIENT_DATA']

canopy_2016['has_string_fault']   = ~canopy_2016['label_string'].isin(NON_FAULT_VALUES)
canopy_2016['has_inverter_fault'] = ~canopy_2016['label_inverter'].isin(NON_FAULT_VALUES)
canopy_2016['has_any_fault']      = canopy_2016['has_string_fault'] | canopy_2016['has_inverter_fault']
canopy_2016['fault_cooccurrence'] = canopy_2016['has_string_fault'] & canopy_2016['has_inverter_fault']

# Summary
print("=== Fault Indicator Summary (2016) ===\n")
print(f"has_string_fault   : {canopy_2016['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2016['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2016['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2016['fault_cooccurrence'].sum()}")

print(f"\nAs % of evaluable rows (Op_Mask=1): "
      f"{canopy_2016.loc[canopy_2016['Op_Mask']==1, 'has_any_fault'].mean() * 100:.2f}%")

=== Fault Indicator Summary (2016) ===

has_string_fault   : 7610
has_inverter_fault : 11428
has_any_fault      : 18442
fault_cooccurrence : 596

As % of evaluable rows (Op_Mask=1): 9.54%


#Stage 1: CS-LSTM Regression Description

In [ ]:
"""
References Paper: "A Comprehensive Framework for Predictive Maintenance in Solar
          Photovoltaic Systems Using Contextual Deep Model"

Objective:
  Predict the expected AC power output (InvPAC_kW_Avg) under normal
  operating conditions by leveraging both operational data and contextual features
  (season and time of day) through Kronecker product expansion

  The residual (measured - expected) that is produced here is used as
  anomaly signal in Stage 2 (XGBoost classifier).

Mathematics:
  Standard LSTM:  gate = σ(W_h·h_{t-1}  +  W_x·x_t           + b)
  CS-LSTM:        gate = σ(W_h·h_{t-1}  +  B·(x_t ⊗ Γ(z_t))  + b)

  where:
    x_t   = primary features (operational data) at timestep t
    z_t   = contextual features (season, time-of-day encodings)
    Γ(z_t) = polynomial basis functions of z_t (degree 2)
    ⊗     = Kronecker product → expanded input vector

  This means that the gate weights of LSTM become functions of
  context - meaning. the model learns that "On July at 12:00 the
  irradiance has a different importance than that of December".
"""

# Hyperparameters (από Baruah et al. 2025 + adaptation)

Info = {
    "train_years":       [2015, 2016],
    "val_year":          2017,
    "test_year":         2018,
    "irr_threshold":     100.0,        # W/m² - operating mask

    # Features
    "primary_features": [              # Inverter signals = x_t
        "POA_fused",                   # POA irradiance
        "AmbTemp_C_Avg",               # ambient temperature
        "module_temp",                 # module temperature
        "InvVDCin_Avg",                # DC voltage
        "InvIDCin_Avg",                # DC current
        "InvPDC_kW_Avg",               # DC power
        "InvPAC_kW_Avg_lag1",          # previous AC power
    ],
    "target":            "InvPAC_kW_Avg",
    "poly_degree":       [2,3],            # polynomial basis degree for Γ(z_t)

    # Model
    "seq_len":           [5,10,15,30,60],           # timesteps per window
    "hidden_size":       [10, 15, 20, 25, 30],       # LSTM hidden units (από Baruah et al.)
    "num_layers":        [1,2],            # LSTM layers (από Baruah et al.)
    "dropout":           [0.1, 0.2, 0.3],

    # Training
    "batch_size":        [64, 128, 256],
    "lr":                [1e-5, 1e-3],       # learning rate (από Baruah et al.)
    "epochs":            [50-150],
    "weight_decay":      1e-6,
}

# Parameters (NIST Canopy)
P_AC_RATED_KW = 260.0   # Inverter rated AC power (kW) PVP260kW
P_DC_NOM_KW   = 243.0   # DC Nameplate power (kW)

In [ ]:
"""
Stage 4:
    Υπολογισμός του Γ(z_t) — polynomial basis functions of z_t.

    Input:
      z      : Τα contextual features (sin_doy, cos_doy, sin_hour, cos_hour)
      degree : polynomial degree 2

    Output:
      basis  :  Πολυωνυμικά γινόμενα μέχρι βαθμού 2

    Παράδειγμα για degree=2, z=[z1, z2]:
      Γ(z) = [1, z1, z2, z1², z1*z2, z2²]

    """

"""Stage 5:

    Υπολογισμός του Kronecker product x_t ⊗ Γ(z_t).

    Input:
      x_t
      Γ(z_t)

    Μαθηματικά:
      [x1, x2] ⊗ [γ1, γ2, γ3] = [x1*γ1, x1*γ2, x1*γ3, x2*γ1, x2*γ2, x2*γ3]

    Αυτό επιτρέπει στο LSTM να έχει ξεχωριστά βαρη για κάθε combination δηλαδή
    η ακτινοβολία έχει διαφορετικό weight ανάλογα με την εποχή και ώρα.
    """

"""
    Stage 5:
    Context-Sensitive LSTM Regression Model

    Αρχιτεκτονική:
      1. Input: x_expanded = x_t ⊗ Γ(z_t)
      2. LSTM layers: εκμάθηση temporal dependencies
      3. FC output layer: πρόβλεψη InvPAC_kW_Avg

    Το μοντέλο εκπαιδεύεται μόνο σε confirmed-normal data.
    Κατά inference, τρέχει σε ολα τα data (normal + faulty)
    για να παράξει residuals.
    """

"""
    Stage 6: Evaluation

    Χρησιμοποιούμε μετρικές όπως RMSE, NRMSE για να κάνουμε την ελαχιστοποιήση των residuals μεταξύ προβλεπόμενης InvPAC_kw_Avg και predicted InvPAC_kW_Avg
    """

#Stage 2: Contexual Features

In [ ]:
"""
Stage 2: Φορτωση μόνο normal data (no fault data)

    We only keep timesteps with confirmed-normal operation for the training of CS-LSTM.

    Criteria:
      1. Op_Mask == 1        → Within operating hours (irradiance threshold)
      2. InvOpStatus_Avg > 0 → inverter ON (no shutdown/idle)
      3. has_any_fault == 0  → no inverter or string faults

    These data are used only for the training part of CS-LSTM to learn the
    "expected normal behavior". During inference it runs on the entire dataset.
    """


In [6]:
def filter_normal(df, label):

    mask = (
        (df['Op_Mask'] == 1) &
        (df['InvOpStatus_Avg'] > 0) &
        (df['has_any_fault'] == False)
    )

    df_normal = df[mask].copy()

    total     = len(df)
    operating = (df['Op_Mask'] == 1).sum()
    normal    = len(df_normal)

    print(f"=== {label} — Normal Data Filter ===")
    print(f"  Total rows:              {total:>10,}")
    print(f"  Operating rows (mask=1): {operating:>10,}  ({100*operating/total:.1f}%)")
    print(f"  Normal rows (no fault):  {normal:>10,}  ({100*normal/operating:.1f}% of operating)")
    print(f"  Excluded (fault/off):    {operating - normal:>10,}")
    print()

    return df_normal

normal_2015 = filter_normal(canopy_2015, "2015")
normal_2016 = filter_normal(canopy_2016, "2016")


=== 2015 — Normal Data Filter ===
  Total rows:                 525,600
  Operating rows (mask=1):    190,478  (36.2%)
  Normal rows (no fault):     169,665  (89.1% of operating)
  Excluded (fault/off):        20,813

=== 2016 — Normal Data Filter ===
  Total rows:                 527,040
  Operating rows (mask=1):    193,285  (36.7%)
  Normal rows (no fault):     174,843  (90.5% of operating)
  Excluded (fault/off):        18,442



In [7]:
# Combining the training set
canopy_train_normal = pd.concat([normal_2015, normal_2016], ignore_index=False)
canopy_train_normal = canopy_train_normal.sort_index()

print(f"=== Combined Training Set (2015-2016) ===")
print(f"  Total normal rows: {len(canopy_train_normal):,}")

=== Combined Training Set (2015-2016) ===
  Total normal rows: 344,508


In [8]:
canopy_train_normal

,TIMESTAMP,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,...,inv_any_fault_reported,inv_fault_count,inv_efficiency_roll5,label_inverter,inv_shutdown_power_flag,inv_shutdown_status_flag,has_string_fault,has_inverter_fault,has_any_fault,fault_cooccurrence
494,2015-01-01 08:14:00-05:00,49.591564,71.623730,110.08793,28.28,69.96,-4.212,377.0,276.6,276.0,...,0,0,0.987993,NORMAL,0,0,False,False,False,False
495,2015-01-01 08:15:00-05:00,49.709785,71.916470,116.14474,28.98,78.30,-4.154,381.1,276.8,275.9,...,0,0,0.985863,NORMAL,0,0,False,False,False,False
496,2015-01-01 08:16:00-05:00,52.901610,77.088220,126.57590,30.37,90.60,-4.082,380.9,276.8,275.9,...,0,0,0.980335,NORMAL,0,0,False,False,False,False
497,2015-01-01 08:17:00-05:00,56.802734,82.747856,135.66109,31.89,98.20,-3.898,381.0,277.0,275.9,...,0,0,0.983207,NORMAL,0,0,False,False,False,False
498,2015-01-01 08:18:00-05:00,59.048836,85.089775,138.46516,33.37,100.90,-3.708,383.6,276.9,276.0,...,0,0,0.981818,NORMAL,0,0,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
526476,2016-12-31 14:36:00-05:00,125.851880,116.777790,124.96544,123.60,122.30,8.100,374.4,276.2,275.0,...,0,0,0.746429,NORMAL,0,0,False,False,False,False
526477,2016-12-31 14:37:00-05:00,121.137120,112.635780,121.04766,119.40,119.00,8.080,374.4,276.1,275.3,...,0,0,0.744774,NORMAL,0,0,False,False,False,False
526478,2016-12-31 14:38:00-05:00,115.029366,107.291260,115.74715,113.90,114.30,8.060,374.2,275.9,275.1,...,0,0,0.741815,NORMAL,0,0,False,False,False,False
526479,2016-12-31 14:39:00-05:00,108.921610,101.679510,110.21617,108.40,109.30,8.030,373.6,276.1,275.2,...,0,0,0.735993,NORMAL,0,0,False,False,False,False


#Stage 3: Primary and Contexual Features

## Excluded Features - Data Leakage

These variables are **never used** as primary features because they either encode the target directly or contain information that would not be available during inference in a real-world deployment.

```text
# Derived directly from InvPAC_kW_Avg — circular dependency
PR, PR_roll_15min_weighted, PR_roll_5min_weighted,
PR_roll_60min_weighted, PR_roll_std_15min,
power_deficit_kW

# Physics-based expected values — the model is supposed to learn this mapping.
# Providing them as inputs would make the regression task trivial.
P_expected_DC_kW, P_expected_AC_kW

# Labels and fault indicators — severe target leakage
label_power, label_string, label_inverter,
has_string_fault, has_inverter_fault, has_any_fault, fault_cooccurrence,
inv_any_fault_reported, inv_fault_count

# Fault flags — also leakage (all zero by definition during normal-operation training)
InvMainFault_Max, InvDriveFault_Max, InvVoltageFault_Max,
InvGridFault_Max, InvTempFault_Max, InvSystemFault_Max, InvSystemWarn_Max

# Operational status flags
InvOpStatus_Avg, Op_Mask, InvPVMStatus_Max,
inv_shutdown_power_flag, inv_shutdown_status_flag, PR_evaluable
```

---

## Excluded Features — Redundancy / Noise

```text
# Pyranometers — POA_fused is already a superior irradiance estimate
Pyra1_Wm2_Avg, Pyra2_Wm2_Avg, Pyra3_Wm2_Avg

# Reference cells — already used to construct POA_fused
RefCell1_Wm2_Avg, RefCell2_Wm2_Avg

# AC power meter measurements — redundant with inverter AC power
# They measure essentially the same quantity at a different point
PwrMtrIa_Avg, PwrMtrIb_Avg, PwrMtrIc_Avg,
PwrMtrFreq_Avg, PwrMtrVa_Avg, PwrMtrVb_Avg, PwrMtrVc_Avg,
PwrMtrP_kW_Avg, PwrMtrPF_Avg

# Three-phase AC measurements — redundant since InvPAC_kW_Avg is already the total AC power
InvVa_Avg, InvVb_Avg, InvVc_Avg,
InvIa_Avg, InvIb_Avg, InvIc_Avg

InvVDVoltage_V_Avg      # DC bus voltage; largely redundant with InvVDCin_Avg
InvVPVin_Avg            # PV input voltage; largely redundant with InvVDCin_Avg
InvFreq_Avg             # Grid frequency; does not determine expected PV power
InvEtot_kWh_Max         # Cumulative energy; not an instantaneous signal

# String-level deviation features — encode fault patterns rather than normal operation
I_string_1_dev .. I_string_7_dev,
I_string_1_rel_dev .. I_string_7_rel_dev,
I_string_1_expected .. I_string_7_expected,
R_string_1_dev .. R_string_7_dev,
Pnorm_string_1 .. Pnorm_string_7,
Inorm_string_1 .. Inorm_string_7,
I_rel_dev_max, I_rel_dev_std, I_rel_dev_max_roll5,
I_rel_dev_max_core, I_rel_dev_max_core_roll5,
I_string_cv_roll5

# Derived inverter-efficiency features
inv_efficiency, inv_efficiency_clipped, inv_efficiency_roll5,
DC_power_discrepancy, DC_power_discrepancy_pct, inv_loading_ratio
```

---

## Primary Features ($x_t$) — 8 Features

These variables closely follow the methodology proposed by **Baruah et al.** and capture the underlying physical behavior of the PV system.

```python
PRIMARY_FEATURES = [
    # Irradiance — primary driver of PV generation
    'POA_fused',            # Plane-of-Array irradiance (W/m²)

    # Temperature — affects module efficiency
    'AmbTemp_C_Avg',        # Ambient temperature (°C)
    'module_temp',          # Module backsheet temperature (°C)

    # DC-side electrical measurements
    'InvVDCin_Avg',         # DC input voltage (V)
    'InvIDCin_Avg',         # DC input current (A)
    'InvPDC_kW_Avg',        # DC power (kW)

    # Array-level summary
    'I_string_mean',        # Mean string current (A)

    # Autoregressive input
    'InvPAC_kW_Avg_lag1',   # Previous AC power (kW), created using shift(1)
]

TARGET = 'InvPAC_kW_Avg'
```

`I_string_mean` is retained because it summarizes the overall operating level of the PV array without explicitly encoding fault information. Unlike string deviation features, the mean current represents normal operating behavior rather than abnormal patterns.

---

## Contextual Features ($z_t$) — 4 Features

These cyclical encodings provide seasonal and diurnal context to the Contextual Sparse LSTM.

```python
CONTEXTUAL_FEATURES = [
    'sin_doy',      # sin(2π × day_of_year / 365)
    'cos_doy',      # cos(2π × day_of_year / 365)
    'sin_hour',     # sin(2π × time_of_day / 24)
    'cos_hour',     # cos(2π × time_of_day / 24)
]
```

Although the original architecture also discusses STL residuals as contextual variables, they are omitted in this implementation. The four cyclical encodings sufficiently capture seasonal variation and time-of-day effects, while STL-based features can be evaluated later as part of an ablation study.

---

## RTD Temperature Sensors — Excluded

```text
RTD_C_Avg_1, RTD_C_Avg_2, RTD_C_Avg_4,
RTD_C_Avg_8, RTD_C_Avg_9, RTD_C_Avg_10
```

These variables correspond to module/backsheet temperatures measured by individual RTD sensors. Since the dataset already includes the reliable `module_temp` measurement (SEWS Module Temperature), the RTD sensors provide little additional information while increasing noise and model complexity. Consequently, they are excluded from the final feature set.

---

    Δημιουργία contextual features z_t:

    Season (εποχή):
      sin_doy = sin(2π × day_of_year / 365)
      cos_doy = cos(2π × day_of_year / 365)
      → Circular encoding που αποφεύγει την ασυνέχεια Dec→Jan

    Time of day (ώρα ημέρας):
      sin_hour = sin(2π × (hour + minute/60) / 24)
      cos_hour = cos(2π × (hour + minute/60) / 24)
      → Circular encoding (23:59 ≈ 00:00)

    z_t που CS-LSTM που χρησιμοποιούνται στο polynomial basis Γ(z_t).

In [10]:
for column in canopy_train_normal.columns:
  print(column)

TIMESTAMP
Pyra1_Wm2_Avg
Pyra2_Wm2_Avg
Pyra3_Wm2_Avg
RefCell1_Wm2_Avg
RefCell2_Wm2_Avg
AmbTemp_C_Avg
InvVDVoltage_V_Avg
InvVa_Avg
InvVb_Avg
InvVc_Avg
InvIa_Avg
InvIb_Avg
InvIc_Avg
InvVDCin_Avg
InvIDCin_Avg
InvFreq_Avg
InvPAC_kW_Avg
InvEtot_kWh_Max
InvVPVin_Avg
InvPDC_kW_Avg
InvOpStatus_Avg
InvMainFault_Max
InvDriveFault_Max
InvVoltageFault_Max
InvGridFault_Max
InvTempFault_Max
InvSystemFault_Max
InvSystemWarn_Max
InvPVMStatus_Max
PwrMtrIa_Avg
PwrMtrIb_Avg
PwrMtrIc_Avg
PwrMtrFreq_Avg
PwrMtrVa_Avg
PwrMtrVb_Avg
PwrMtrVc_Avg
PwrMtrP_kW_Avg
PwrMtrPF_Avg
ShuntCurrent_A_Avg_1
ShuntCurrent_A_Avg_2
ShuntCurrent_A_Avg_3
ShuntCurrent_A_Avg_4
ShuntCurrent_A_Avg_5
ShuntCurrent_A_Avg_6
ShuntCurrent_A_Avg_7
ShuntPDC_kW_Avg_1
ShuntPDC_kW_Avg_2
ShuntPDC_kW_Avg_3
ShuntPDC_kW_Avg_4
ShuntPDC_kW_Avg_5
ShuntPDC_kW_Avg_6
ShuntPDC_kW_Avg_7
RTD_C_Avg_1
RTD_C_Avg_2
RTD_C_Avg_4
RTD_C_Avg_8
RTD_C_Avg_9
RTD_C_Avg_10
Op_Mask
POA_fused
PR_evaluable
module_temp
P_expected_DC_kW
P_expected_AC_kW
PR
PR_roll_15min_weight

In [13]:
# Feature Selection & Contextual Encoding

PRIMARY_FEATURES = [
    'POA_fused',
    'AmbTemp_C_Avg',
    'module_temp',
    'InvVDCin_Avg',
    'InvIDCin_Avg',
    'InvPDC_kW_Avg',
    'I_string_mean',
    'InvPAC_kW_Avg_lag1',
]

CONTEXTUAL_FEATURES = [
    'sin_doy',
    'cos_doy',
    'sin_hour',
    'cos_hour',
]

TARGET = 'InvPAC_kW_Avg'


def add_features(df):
    """
    Adds the lag feature and contextual encodings.

    This function is applied to both the training set and any other dataset
    (validation, test, or inference).

    NOTE:
    The lag feature is computed before splitting the data into windows,
    ensuring that the previous time step is correctly preserved,
    including across day boundaries.
    """
    df = df.copy()

    # Autoregressive feature
    df['InvPAC_kW_Avg_lag1'] = df['InvPAC_kW_Avg'].shift(1)

    # Contextual features
    if 'TIMESTAMP' in df.columns:
      df = df.set_index(pd.to_datetime(df['TIMESTAMP']))

    doy  = df.index.day_of_year.values
    hour = df.index.hour.values + df.index.minute.values / 60.0

    df['sin_doy']  = np.sin(2 * np.pi * doy  / 365.0)
    df['cos_doy']  = np.cos(2 * np.pi * doy  / 365.0)
    df['sin_hour'] = np.sin(2 * np.pi * hour / 24.0)
    df['cos_hour'] = np.cos(2 * np.pi * hour / 24.0)

    return df


canopy_train_normal = add_features(canopy_train_normal)

# Remove NaN values created from lag (first row of each continuity group)
required_cols = PRIMARY_FEATURES + CONTEXTUAL_FEATURES + [TARGET]
canopy_train_normal = canopy_train_normal.dropna(subset=required_cols)

print(f"Training rows after feature engineering: {len(canopy_train_normal):,}")
print(f"\nPrimary features ({len(PRIMARY_FEATURES)}):")

for f in PRIMARY_FEATURES:
    print(f"  {f:30s}  mean={canopy_train_normal[f].mean():.3f}  "
          f"std={canopy_train_normal[f].std():.3f}")

print(f"\nContextual features ({len(CONTEXTUAL_FEATURES)}):")

for f in CONTEXTUAL_FEATURES:
    print(f"  {f:30s}  range=[{canopy_train_normal[f].min():.2f}, "
          f"{canopy_train_normal[f].max():.2f}]")

print(f"\nTarget: {TARGET}")
print(f"  mean={canopy_train_normal[TARGET].mean():.3f} kW  "
      f"max={canopy_train_normal[TARGET].max():.3f} kW")

Training rows after feature engineering: 344,483

Primary features (8):
  POA_fused                       mean=435.509  std=263.041
  AmbTemp_C_Avg                   mean=18.568  std=9.859
  module_temp                     mean=30.175  std=15.047
  InvVDCin_Avg                    mean=347.831  std=23.591
  InvIDCin_Avg                    mean=316.660  std=189.812
  InvPDC_kW_Avg                   mean=107.330  std=60.288
  I_string_mean                   mean=41.656  std=25.687
  InvPAC_kW_Avg_lag1              mean=95.909  std=55.696

Contextual features (4):
  sin_doy                         range=[-1.00, 1.00]
  cos_doy                         range=[-1.00, 1.00]
  sin_hour                        range=[-1.00, 1.00]
  cos_hour                        range=[-1.00, 0.25]

Target: InvPAC_kW_Avg
  mean=95.912 kW  max=260.000 kW


#Stage 4: Window handling

In [37]:
# Continuity Groups

def assign_continuity_groups(df: pd.DataFrame,
                              max_gap_minutes: int = 1) -> pd.DataFrame:
    """
    Ανιχνεύει ασυνέχειες στο DatetimeIndex και αναθέτει group ID.

    Δύο διαδοχικές γραμμές ανήκουν στο ίδιο group αν:
      - Η χρονική διαφορά τους είναι ακριβώς 1 λεπτό (max_gap_minutes)

    Οποιοδήποτε gap > 1 λεπτό (νύχτα, fault period, missing data,
    inverter shutdown) ξεκινά νέο group.

    Παράδειγμα:
      07:00 → 18:59  (group 0)
      19:00 → gap →
      07:00 → 18:59  (group 1)  ← νέα μέρα, νέο group
    """
    df = df.copy()
    time_diff = df.index.to_series().diff()
    gap_mask  = time_diff > pd.Timedelta(minutes=max_gap_minutes)
    df['continuity_group'] = gap_mask.cumsum()

    group_sizes = df.groupby('continuity_group').size()
    print(f"Continuity groups: {len(group_sizes):,}")
    print(f"Group size stats (timesteps):")
    print(f"  min:    {group_sizes.min():>6,}")
    print(f"  median: {group_sizes.median():>6,.0f}")
    print(f"  max:    {group_sizes.max():>6,}")
    print(f"  Groups ≥ seq_len ({SEQ_LEN}): "
          f"{(group_sizes >= SEQ_LEN).sum():,}  "
          f"(usable for windowing)")

    return df


SEQ_LEN = 60  # 60 λεπτά look-back window

canopy_train_normal = assign_continuity_groups(canopy_train_normal)

Continuity groups: 90,657
Group size stats (timesteps):
  min:         1
  median:      3
  max:       387
  Groups ≥ seq_len (60): 205  (usable for windowing)


#Model

In [38]:
# Dataset & DataLoader

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import numpy as np


class CSLSTMDataset(Dataset):
    def __init__(
        self,
        df,
        primary_cols,
        context_cols,
        target_col,
        seq_len,
        scaler_x=None,
        scaler_y=None,
        fit_scaler=False,
    ):

        self.seq_len = seq_len

        # -----------------------------
        # Raw arrays
        # -----------------------------
        X_raw = df[primary_cols].values.astype(np.float32)
        Z_raw = df[context_cols].values.astype(np.float32)
        Y_raw = df[target_col].values.astype(np.float32)

        groups = df["continuity_group"].values

        # -----------------------------
        # Feature scaling
        # -----------------------------
        if fit_scaler:
            self.scaler_x = StandardScaler()
            self.scaler_y = StandardScaler()

            self.scaler_x.fit(X_raw)
            self.scaler_y.fit(Y_raw.reshape(-1, 1))

        else:
            self.scaler_x = scaler_x
            self.scaler_y = scaler_y

        X_scaled = self.scaler_x.transform(X_raw)

        Y_scaled = (
            self.scaler_y
            .transform(Y_raw.reshape(-1, 1))
            .flatten()
        )

        # -----------------------------
        # Sliding windows
        # -----------------------------
        X_wins = []
        Z_wins = []
        Y_tgts = []

        skipped_small = 0

        for g in np.unique(groups):

            idx = np.where(groups == g)[0]

            if len(idx) < seq_len:
                skipped_small += 1
                continue

            X_g = X_scaled[idx]
            Z_g = Z_raw[idx]
            Y_g = Y_scaled[idx]

            for i in range(len(idx) - seq_len + 1):

                X_wins.append(X_g[i:i + seq_len])
                Z_wins.append(Z_g[i:i + seq_len])
                Y_tgts.append(Y_g[i + seq_len - 1])

        self.X = np.asarray(X_wins, dtype=np.float32)
        self.Z = np.asarray(Z_wins, dtype=np.float32)
        self.Y = np.asarray(Y_tgts, dtype=np.float32)

        # -----------------------------
        # Summary
        # -----------------------------
        print("=== CS-LSTM Dataset Summary ===")
        print(f"Windows created:            {len(self.X):>10,}")
        print(f"Skipped small groups:       {skipped_small:>10,}")
        print()
        print(f"Primary input shape : {self.X.shape}")
        print(f"Context input shape : {self.Z.shape}")
        print(f"Target shape        : {self.Y.shape}")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        return (
            torch.from_numpy(self.X[idx]),      # (seq_len, n_primary)
            torch.from_numpy(self.Z[idx]),      # (seq_len, n_context)
            torch.tensor(self.Y[idx]),          # scalar target
        )

In [39]:
BATCH_SIZE = 128


train_ds = CSLSTMDataset(
    df=canopy_train_normal,
    primary_cols=PRIMARY_FEATURES,
    context_cols=CONTEXTUAL_FEATURES,
    target_col=TARGET,
    seq_len=SEQ_LEN,
    fit_scaler=True
)


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


print(
    f"Train batches: {len(train_loader):,}"
)

=== CS-LSTM Dataset Summary ===
Windows created:                12,025
Skipped small groups:           90,452

Primary input shape : (12025, 60, 8)
Context input shape : (12025, 60, 4)
Target shape        : (12025,)
Train batches: 94


In [40]:
# ============================================
# Validation Data
# ============================================

canopy_2017 = pd.read_csv(
    "/content/drive/MyDrive/NIST_Preprocessed/canopy_2017_Labeled.csv"
)


# Fault indicators

canopy_2017['has_string_fault'] = (
    ~canopy_2017['label_string']
    .isin(NON_FAULT_VALUES)
)


canopy_2017['has_inverter_fault'] = (
    ~canopy_2017['label_inverter']
    .isin(NON_FAULT_VALUES)
)


canopy_2017['has_any_fault'] = (
    canopy_2017['has_string_fault'] |
    canopy_2017['has_inverter_fault']
)



# Keep operating data only

df_val = canopy_2017[
    canopy_2017['Op_Mask']==1
].copy()


df_val = add_features(df_val)


df_val = df_val.dropna(
    subset=PRIMARY_FEATURES+
           CONTEXTUAL_FEATURES+
           [TARGET]
)


df_val = df_val.sort_index()


df_val = assign_continuity_groups(
    df_val,
    max_gap_minutes=1
)


print(
    f"Validation rows: {len(df_val):,}"
)

print(
    f"Fault rows: {df_val.has_any_fault.sum():,}"
)

Continuity groups: 1,384
Group size stats (timesteps):
  min:         1
  median:     12
  max:       792
  Groups ≥ seq_len (60): 403  (usable for windowing)
Validation rows: 191,955
Fault rows: 22,951


In [41]:
# ============================================
# Validation Dataset
# ============================================


val_ds = CSLSTMDataset(
    df=df_val,
    primary_cols=PRIMARY_FEATURES,
    context_cols=CONTEXTUAL_FEATURES,
    target_col=TARGET,
    seq_len=SEQ_LEN,
    scaler_x=train_ds.scaler_x,
    scaler_y=train_ds.scaler_y,
    fit_scaler=False
)


val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE*2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

=== CS-LSTM Dataset Summary ===
Windows created:               157,291
Skipped small groups:              981

Primary input shape : (157291, 60, 8)
Context input shape : (157291, 60, 4)
Target shape        : (157291,)


In [42]:
# ============================================
# CS-LSTM Model
# ============================================

import torch.nn as nn
from math import comb
from itertools import combinations_with_replacement


class CSLSTM(nn.Module):

    def __init__(
        self,
        n_primary,
        n_context,
        hidden_size,
        num_layers,
        dropout,
        poly_degree
    ):

        super().__init__()

        self.poly_degree = poly_degree

        basis_size = comb(
            n_context + poly_degree,
            poly_degree
        )

        input_size = (
            n_primary *
            basis_size
        )


        print("CS-LSTM dimensions")
        print("------------------")
        print(f"Primary features : {n_primary}")
        print(f"Context features : {n_context}")
        print(f"Polynomial basis : {basis_size}")
        print(f"LSTM input size  : {input_size}")


        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=(
                dropout
                if num_layers > 1
                else 0
            )
        )


        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_size,
            1
        )


    def poly_basis(self,z):

        B,T,N = z.shape

        terms = [
            torch.ones(
                B,T,1,
                device=z.device
            )
        ]


        for d in range(
            1,
            self.poly_degree+1
        ):

            for combo in combinations_with_replacement(
                range(N),
                d
            ):

                term = torch.ones(
                    B,T,1,
                    device=z.device
                )

                for i in combo:
                    term *= z[:,:,i:i+1]

                terms.append(term)


        return torch.cat(
            terms,
            dim=-1
        )


    def forward(self,x,z):

        gamma = self.poly_basis(z)


        x_exp = (
            x.unsqueeze(-1)
            *
            gamma.unsqueeze(-2)
        )


        x_exp = x_exp.flatten(
            start_dim=-2
        )


        out,_ = self.lstm(x_exp)


        last = self.dropout(
            out[:,-1,:]
        )


        return self.fc(last).squeeze(-1)

In [43]:
# ============================================
# Initialize Model
# ============================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


model = CSLSTM(
    n_primary=len(PRIMARY_FEATURES),
    n_context=len(CONTEXTUAL_FEATURES),
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    poly_degree=2
)


model.to(device)

CS-LSTM dimensions
------------------
Primary features : 8
Context features : 4
Polynomial basis : 15
LSTM input size  : 120


CSLSTM(
  (lstm): LSTM(120, 128, num_layers=2, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [44]:
# ============================================
# Shape Check
# ============================================

x,z,y = next(iter(train_loader))

x = x.to(device)
z = z.to(device)

prediction = model(x,z)


print("X:", x.shape)
print("Z:", z.shape)
print("Prediction:", prediction.shape)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


X: torch.Size([128, 60, 8])
Z: torch.Size([128, 60, 4])
Prediction: torch.Size([128])


#Training

In [ ]:
# ============================================
# Training Setup
# ============================================

import torch
import torch.nn as nn
import numpy as np

from sklearn.metrics import r2_score


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

In [ ]:
# ============================================
# Initialize Model
# ============================================

model = CSLSTM(
    n_primary=len(PRIMARY_FEATURES),
    n_context=len(CONTEXTUAL_FEATURES),
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    poly_degree=2
)


model = model.to(DEVICE)

In [ ]:
# ============================================
# Training Utilities
# ============================================

from sklearn.metrics import r2_score


def train_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0


    for x, z, y in loader:

        x = x.to(device)
        z = z.to(device)
        y = y.to(device)


        optimizer.zero_grad()


        pred = model(
            x,
            z
        )


        loss = criterion(
            pred,
            y
        )


        loss.backward()


        # Gradient clipping
        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        optimizer.step()


        total_loss += (
            loss.item()
            *
            len(y)
        )


    return (
        total_loss /
        len(loader.dataset)
    )



@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
    scaler_y
):

    model.eval()


    all_preds = []
    all_true  = []


    for x, z, y in loader:

        x = x.to(device)
        z = z.to(device)


        pred = model(
            x,
            z
        )


        all_preds.append(
            pred.cpu().numpy()
        )

        all_true.append(
            y.numpy()
        )


    preds_s = np.concatenate(
        all_preds
    )

    true_s = np.concatenate(
        all_true
    )


    loss = criterion(
        torch.tensor(preds_s),
        torch.tensor(true_s)
    ).item()



    # --------------------------
    # Convert back to kW
    # --------------------------

    preds_kw = scaler_y.inverse_transform(
        preds_s.reshape(-1,1)
    ).flatten()


    true_kw = scaler_y.inverse_transform(
        true_s.reshape(-1,1)
    ).flatten()



    rmse = np.sqrt(
        np.mean(
            (preds_kw - true_kw)**2
        )
    )


    nrmse = (
        rmse /
        true_kw.mean()
    )


    mae = np.mean(
        np.abs(
            preds_kw - true_kw
        )
    )


    r2 = r2_score(
        true_kw,
        preds_kw
    )


    return (
        loss,
        rmse,
        nrmse,
        r2,
        mae
    )

In [ ]:
# ============================================
# Optimizer
# ============================================


EPOCHS = 100

LR = 0.0009

PATIENCE = 10


optimizer = torch.optim.RMSprop(
    model.parameters(),
    lr=LR,
    weight_decay=1e-6
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5
)


criterion = nn.MSELoss()

In [ ]:
# ============================================
# Training Loop
# ============================================


history = {
    "train_loss": [],
    "val_loss": [],
    "val_rmse": [],
    "val_r2": []
}


best_val_loss = float("inf")

best_weights = None

patience_counter = 0



print(
    f"{'Epoch':>5} | "
    f"{'Train Loss':>10} | "
    f"{'Val Loss':>10} | "
    f"{'RMSE(kW)':>10} | "
    f"{'NRMSE':>7} | "
    f"{'R²':>7}"
)

print("-"*65)



for epoch in range(1, EPOCHS+1):


    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        DEVICE
    )


    val_loss, rmse, nrmse, r2, mae = evaluate(
        model,
        val_loader,
        criterion,
        DEVICE,
        train_ds.scaler_y
    )


    scheduler.step(
        val_loss
    )



    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_rmse"].append(
        rmse
    )

    history["val_r2"].append(
        r2
    )



    # -----------------------
    # Early stopping
    # -----------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_weights = {
            k:v.clone()
            for k,v in model.state_dict().items()
        }

        patience_counter = 0

        flag = "✓"


    else:

        patience_counter += 1

        flag = (
            f"({patience_counter}/{PATIENCE})"
        )



    if epoch == 1 or epoch % 5 == 0:

        print(
            f"{epoch:>5} | "
            f"{train_loss:>10.5f} | "
            f"{val_loss:>10.5f} | "
            f"{rmse:>10.3f} | "
            f"{nrmse:>7.4f} | "
            f"{r2:>7.4f} {flag}"
        )


    if patience_counter >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}"
        )

        break

In [ ]:
# ============================================
# Restore Best Model
# ============================================


model.load_state_dict(
    best_weights
)


print(
    f"Best validation loss: {best_val_loss:.6f}"
)